In [5]:
import pandas as pd
from urllib.request import urlopen  
import os.path as osp
import os
import logging
import zipfile
from glob import glob
logging.getLogger().setLevel('INFO')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

## Helpers

In [6]:
def download_file(url_str, path):
    url = urlopen(url_str)
    output = open(path, 'wb')       
    output.write(url.read())
    output.close()  
    
def extract_file(archive_path, target_dir):
    zip_file = zipfile.ZipFile(archive_path, 'r')
    zip_file.extractall(target_dir)
    zip_file.close()

## Download the dataset

In [7]:
BASE_URL = 'http://tennis-data.co.uk'
DATA_DIR = "tennis_data"
ATP_DIR = './{}/ATP'.format(DATA_DIR)
WTA_DIR = './{}/WTA'.format(DATA_DIR)

ATP_URLS = [BASE_URL + "/%i/%i.zip" % (i,i) for i in range(2000,2019)]
WTA_URLS = [BASE_URL + "/%iw/%i.zip" % (i,i) for i in range(2007,2019)]

os.makedirs(osp.join(ATP_DIR, 'archives'), exist_ok=True)
os.makedirs(osp.join(WTA_DIR, 'archives'), exist_ok=True)

for files, directory in ((ATP_URLS, ATP_DIR), (WTA_URLS, WTA_DIR)):
    for dl_path in files:
        logging.info("downloading & extracting file %s", dl_path)
        archive_path = osp.join(directory, 'archives', osp.basename(dl_path))
        download_file(dl_path, archive_path)
        extract_file(archive_path, directory)
    
ATP_FILES = sorted(glob("%s/*.xls*" % ATP_DIR))
WTA_FILES = sorted(glob("%s/*.xls*" % WTA_DIR))

df_atp = pd.concat([pd.read_excel(f) for f in ATP_FILES], ignore_index=True)
df_wta = pd.concat([pd.read_excel(f) for f in WTA_FILES], ignore_index=True)

logging.info("%i matches ATP in df_atp", df_atp.shape[0])
logging.info("%i matches WTA in df_wta", df_wta.shape[0])

INFO:root:downloading & extracting file http://tennis-data.co.uk/2000/2000.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2001/2001.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2002/2002.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2003/2003.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2004/2004.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2005/2005.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2006/2006.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2007/2007.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2008/2008.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2009/2009.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2010/2010.zip
INFO:root:downloading & extracting file http://tennis-data.co.uk/2011/2011.zip
INFO:root:downloading & extracting file http://tenni

## Your work

Tennis match outcome prediction (ATP)

In this notebook, we use historical ATP match data from tennis-data.co.uk (2000–2018) to:
- answer four introductory questions about the dataset,
- build features describing each match (ranking, surface, past winrates, etc.),
- train two models (Logistic Regression and Random Forest),
- evaluate their performance on matches played in 2017.

The goal is not to build the most sophisticated model possible, but to have
a clean, well-reasoned pipeline and interpretable results.

In [ ]:

# =====================================================
# Top 3 ATP players with the most wins
# =====================================================
print("=== Q1: Top 3 ATP players with the most wins ===")
print(df_atp['Winner'].value_counts().head(3), "\n")

# =====================================================
# Total number of sets won by Federer R.  
# =====================================================
print("=== Q2: Total number of sets won by Federer R. ===")
print(df_atp[df_atp['Winner'] == 'Federer R.']['Wsets'].sum(), "\n")

# =====================================================
# Number of sets won by Federer R. in 2016 and 2017
# =====================================================
df_atp['Date'] = pd.to_datetime(df_atp['Date'], errors='coerce')
df_atp['Year'] = df_atp['Date'].dt.year

print("=== Q3: Sets won by Federer R. in 2016 and 2017 ===")
print(df_atp[(df_atp['Winner'] == 'Federer R.') & (df_atp['Year'].isin([2016, 2017]))]['Wsets'].sum())

'''
To prepare data for modeling, we:
- clean ranking and points columns (WRank, LRank, WPts, LPts),
- compute past winrates for each player before each match (Q4),
- define a canonical match representation: Player1 vs Player2,
- build numeric features describing the matchup.
'''
# Fix ranking and points columns
for col in ['WRank', 'LRank']:
    # If some ranks are missing, treat them as unranked with a big value
    df_atp[col] = pd.to_numeric(df_atp[col], errors='coerce').fillna(9999)

for col in ['WPts', 'LPts']:
    df_atp[col] = pd.to_numeric(df_atp[col], errors='coerce').fillna(0)

# =====================================================
# For each match: winner's past win percentage
# (and also loser's, for later features)
# =====================================================
df_atp = df_atp.sort_values('Date').reset_index(drop=True)

player_stats = {}          # player -> {'wins': x, 'matches': y}
winner_past = []           # winner's past win ratio
loser_past = []            # loser's past win ratio

for _, row in df_atp.iterrows():
    winner = row['Winner']
    loser  = row['Loser']
    
    # init if first time we see the player
    for p in (winner, loser):
        if p not in player_stats:
            player_stats[p] = {'wins': 0, 'matches': 0}
    
    w_stats = player_stats[winner]
    l_stats = player_stats[loser]
    
    # winner's past win ratio BEFORE this match
    if w_stats['matches'] == 0:
        # No prior matches for this player, so we keep NaN
        winner_past.append(np.nan)
    else:
        winner_past.append(w_stats['wins'] / w_stats['matches'])
    
    # loser's past win ratio BEFORE this match
    if l_stats['matches'] == 0:
        # No prior matches for this player, so we keep NaN
        loser_past.append(np.nan)
    else:
        loser_past.append(l_stats['wins'] / l_stats['matches'])
    
    # update stats AFTER the match
    player_stats[winner]['wins']    += 1
    player_stats[winner]['matches'] += 1
    player_stats[loser]['matches']  += 1

df_atp['WinnerPastWinRatio'] = winner_past
df_atp['LoserPastWinRatio']  = loser_past
df_atp['WinnerPastWinPct'] = (df_atp['WinnerPastWinRatio'] * 100).round(2)
df_atp['LoserPastWinPct']  = (df_atp['LoserPastWinRatio']  * 100).round(2)

print("=== Q4: Winner's past win percentage (first 100 matches) ===")
display(df_atp[['Date', 'Winner', 'WinnerPastWinPct']].head(100))

# =====================================================
# Data preparation for the modeling part
# =====================================================

# 1) Date & Year (already done)
#df_atp['Date'] = pd.to_datetime(df_atp['Date'], errors='coerce')
#df_atp['Year'] = df_atp['Date'].dt.year

# 2) Remove matches without proper players and remove walkovers
df_atp = df_atp.dropna(subset=['Winner', 'Loser'])
df_atp = df_atp[df_atp['Comment'] != 'Walkover']

# 3) Build (Player1, Player2, y) where Player1 is alphabetical
'''
Canonical match representation to avoids having two versions of the same match:
- Player1 and Player2 as the two opponents,
- with Player1 chosen as the player whose name is alphabetically smaller,
- y = 1 if Player1 is the winner, y = 0 otherwise.
'''
def build_match_row(row):
    w = row['Winner']
    l = row['Loser']
    
    # Order players alphabetically
    if w < l:
        p1, p2 = w, l
        y = 1   # winner is Player1
    else:
        p1, p2 = l, w
        y = 0   # winner is Player2
    
    return pd.Series([p1, p2, y])

df_matches = df_atp.apply(build_match_row, axis=1)
df_matches.columns = ['Player1', 'Player2', 'y']

# Make sure indices are aligned before concatenation
df_atp = df_atp.reset_index(drop=True)
df_matches = df_matches.reset_index(drop=True)

# 4) Extract features for each match, aligned with df_matches and df_atp
'''
For each match we create:

- rank1, rank2, rank_diff (ATP rankings),
- pts1, pts2, pts_diff (ATP points),
- winrate1, winrate2, winrate_diff (historical winrates),
- surface (one-hot encoded),
- best_of (3 or 5),
- year (for the train/test split).
'''
def extract_features(row_match, row_atp):
    p1 = row_match['Player1']
    p2 = row_match['Player2']
    w  = row_atp['Winner']
    l  = row_atp['Loser']
    
    out = {}
    
    # Rankings, points, past winrates mapped to Player1 / Player2
    if p1 == w:
        out['rank1'] = row_atp['WRank']
        out['rank2'] = row_atp['LRank']
        out['pts1'] = row_atp['WPts']
        out['pts2'] = row_atp['LPts']
        out['winrate1'] = row_atp['WinnerPastWinRatio']
        out['winrate2'] = row_atp['LoserPastWinRatio']
    else:
        out['rank1'] = row_atp['LRank']
        out['rank2'] = row_atp['WRank']
        out['pts1'] = row_atp['LPts']
        out['pts2'] = row_atp['WPts']
        out['winrate1'] = row_atp['LoserPastWinRatio']
        out['winrate2'] = row_atp['WinnerPastWinRatio']
    
    # Differences
    out['rank_diff'] = out['rank2'] - out['rank1']
    out['pts_diff'] = out['pts2'] - out['pts1']
    out['winrate_diff'] = out['winrate2'] - out['winrate1']
    
    # Other fields
    out['surface'] = row_atp['Surface']
    out['best_of'] = row_atp['Best of']
    out['year'] = row_atp['Year']
    
    return pd.Series(out)

features_list = []
for i in range(len(df_atp)):
    features_list.append(
        extract_features(df_matches.iloc[i], df_atp.iloc[i])
    )

df_features = pd.DataFrame(features_list)

# 5) Build full modeling dataframe
df_model = pd.concat([df_matches, df_features], axis=1)

# 6) One-hot encode surface
df_model = pd.get_dummies(df_model, columns=['surface'], drop_first=True)


# 7) Train / Test split by year
'''
We train on all matches up to 2016 and evaluate on 2017:
- Train: 2000–2016  
- Test: 2017 only  
'''
train = df_model[df_model['year'] <= 2016]
test = df_model[df_model['year'] == 2017]

X_train = train.drop(columns=['Player1', 'Player2', 'y', 'year'])
y_train = train['y']

X_test = test.drop(columns=['Player1', 'Player2', 'y', 'year'])
y_test = test['y']

def fill_missing_features(X_train, X_test):
    # If some ranks are missing, treat them as unranked with a big value
    for col in ['rank1', 'rank2', 'rank_diff']:
        if col in X_train.columns:
            X_train[col] = X_train[col].fillna(9999)
            X_test[col]  = X_test[col].fillna(9999)
    
    # If some winrates are missing (new players), we can set them to 0.5 (neutral)
    for col in ['winrate1', 'winrate2', 'winrate_diff']:
        if col in X_train.columns:
            X_train[col] = X_train[col].fillna(0.5)
            X_test[col]  = X_test[col].fillna(0.5)

    # Any remaining NaNs
    X_train = X_train.fillna(0)
    X_test  = X_test.fillna(0)
    
    return X_train, X_test

# We use copies to avoid surprises
X_train_lr, X_test_lr = fill_missing_features(X_train.copy(), X_test.copy())

# ============================================================
# Logistic Regression model (baseline)
# ============================================================
'''
We start with a Logistic Regression, which is:
- well-suited to binary classification,
- simple and fast,
- easy to interpret (coefficients show how each feature affects win probability).

We standardize the features and train the model on the 2000–2016 matches.
'''

logreg_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        solver='lbfgs'
    ))
])

logreg_clf.fit(X_train_lr, y_train)

# Predictions
y_pred_train = logreg_clf.predict(X_train_lr)
y_pred_test = logreg_clf.predict(X_test_lr)

# Probabilities (for analysis later)
y_proba_test = logreg_clf.predict_proba(X_test_lr)[:, 1]

# Accuracy
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test,  y_pred_test)

print(f"Logistic Regression - Train accuracy: {train_acc:.3f}")
print(f"Logistic Regression - Test accuracy (2017): {test_acc:.3f}")

baseline = y_train.mode()[0]
baseline_acc = (y_test == baseline).mean()
print(f"Baseline accuracy (always predict {baseline}): {baseline_acc:.3f}")

print("Classification report (test 2017):")
print(classification_report(y_test, y_pred_test, target_names=['Player1 loses', 'Player1 wins']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test)
plt.title("Logistic Regression - Confusion matrix (test 2017)")
plt.show()

# Get feature names and coefficients
feature_names = X_train_lr.columns
coefs = logreg_clf.named_steps['logreg'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
}).sort_values('coef', ascending=False)

coef_df.head(10)
coef_df.tail(10)

# Build a small DataFrame for plotting
plot_df = pd.DataFrame({
    'rank_diff': X_test_lr['rank_diff'],
    'proba_lr':  y_proba_test,
    'y_true':    y_test.values
})

# Bin rank_diff into intervals
bins = np.linspace(plot_df['rank_diff'].min(), plot_df['rank_diff'].max(), 20)
plot_df['rank_bin'] = pd.cut(plot_df['rank_diff'], bins)

grouped = plot_df.groupby('rank_bin').agg(
    mean_rank_diff=('rank_diff', 'mean'),
    mean_proba=('proba_lr', 'mean'),
    empirical_win_rate=('y_true', 'mean'),
    count=('y_true', 'size')
).dropna()

plt.figure(figsize=(8, 5))
plt.plot(grouped['mean_rank_diff'], grouped['mean_proba'], label='Predicted win prob (LR)')
plt.plot(grouped['mean_rank_diff'], grouped['empirical_win_rate'], linestyle='--', label='Empirical win rate')
plt.xlabel('Rank difference (Player2 - Player1)')
plt.ylabel('Probability that Player1 wins')
plt.title('Logistic Regression – Win probability vs rank difference (test 2017)')
plt.legend()
plt.tight_layout()
plt.show()

'''
# Logistic Regression results:
- Test accuracy (2017): around 65%
- Baseline (always predicting the majority class): around 51%

Both classes (Player1 wins / loses) have similar precision and recall,
so the model is not strongly biased toward one outcome.

We plot Win probability vs rank difference which shows how the model's predicted probability 
that Player1 wins changes with the rank difference (Player2 rank − Player1 rank).
- If rank_diff < 0 → Player1 is better ranked → win probability increases  
- If rank_diff > 0 → Player2 is better ranked → Player1 win probability decreases  

The model's curve follows the same trend as the empirical win rate in the data, which indicates that:
- the model is well-calibrated,
- it correctly captures the most important tennis pattern: better-ranked players are more likely to win.
'''

# ============================================================
# Random Forest model
# ============================================================
'''
We train a Random Forest classifier to capture possible nonlinear interactions between features.

We use 300 trees with a fixed random seed for reproducibility.
'''

rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train_lr, y_train)

# Predictions
y_pred_train_rf = rf_clf.predict(X_train_lr)
y_pred_test_rf  = rf_clf.predict(X_test_lr)

# Probabilities
y_proba_test_rf = rf_clf.predict_proba(X_test_lr)[:, 1]

# Accuracy
train_acc_rf = accuracy_score(y_train, y_pred_train_rf)
test_acc_rf  = accuracy_score(y_test,  y_pred_test_rf)

print(f"Random Forest - Train accuracy: {train_acc_rf:.3f}")
print(f"Random Forest - Test accuracy (2017): {test_acc_rf:.3f}")

print("Random Forest - Classification report (test 2017):")
print(classification_report(y_test, y_pred_test_rf, target_names=['Player1 loses', 'Player1 wins']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test_rf)
plt.title("Random Forest - Confusion matrix (test 2017)")
plt.show()

importances = rf_clf.feature_importances_
feature_names = X_train_lr.columns

fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

fi_df.head(15)

top_n = 15
top_features = fi_df.head(top_n)

plt.figure(figsize=(8, 6))
plt.barh(top_features['feature'][::-1], top_features['importance'][::-1])
plt.xlabel("Feature importance")
plt.title("Random Forest - Top features")
plt.tight_layout()
plt.show()

'''
# Random Forest results:
- Train accuracy: 100% (the model perfectly fits the training data)
- Test accuracy (2017): around 64%

The Random Forest slightly underperforms the Logistic Regression on the test set,
suggesting that the main signal is essentially linear in the engineered features.
Feature importance analysis (see plot below) confirms that ranking and past winrate
are the dominant predictors.

We plot also Top Features bar that shows the features that the Random Forest model considers
most important when predicting match outcomes. We observe that:
- winrate_diff and rank_diff are the strongest predictors, confirming that ranking and past performance drive match outcomes.
- surface and best_of have a moderate impact, reflecting differences in playing conditions.
- Other features contribute less but still help refine the prediction.

Overall, the model relies on intuitive tennis factors: players with better
rankings and stronger historical results are more likely to win.
'''

# Conclusion and possible improvements

Summary :
- We built a complete pipeline to predict ATP match outcomes for 2017 based on historical data.
- Features include ranking, ATP points, past winrates, surface and match format.
- Logistic Regression achieves about 65% accuracy on 2017, clearly above the 51% baseline.
- A Random Forest achieves about 64% accuracy but strongly overfits the training set.

Interpretation:
- Although Random Forests can model nonlinear patterns, our engineered features (rank difference, winrate difference, points, surface) have a mostly linear and monotonic relationship with match outcomes. Logistic Regression captures this structure very efficiently.
- The Random Forest, on the other hand, overfits the training data (100% accuracy) but does not generalize better, because the underlying signal is simple and somewhat noisy.
This explains why Logistic Regression achieves slightly higher test accuracy despite the theoretical nonlinearity of tennis outcomes.

Features:
- Ranking difference and past winrate difference are the most informative features,
  which matches tennis intuition: better-ranked and historically more successful players
  are more likely to win.
- Surface has a moderate effect, while other features play a smaller role.

Limitations:
- We did not use bookmaker odds, which are known to be very predictive.
- Past performance is summarized by a simple global winrate; more refined
  metrics (Elo ratings, surface-specific winrates, recent form) could improve accuracy.
- No hyperparameter tuning or cross-validation was performed.

Possible improvements:
- Introduce Elo or Glicko ratings that update after each match.
- Separate winrates by surface (hard/clay/grass).
- Include head-to-head statistics between players.
- Perform cross-validation and hyperparameter tuning for the Random Forest.